# Founder haplotypes

`xftsim` simulations can be highly realistic, but they can also be
quite simplistic. We can use real phased haplotype data or simulate
haplotypes as independent Bernoulli trials. As fully synthetic data is
extremely convenient to work with, we recommend at the very least
using such data to prototype or debug simulations.

In what follows we first introduce tools for generating haplotypes
from scratch, then tools for importing external haplotype data.

## Haplotypes from scratch

The simplest founder constructors generate independent Bernoulli
draws. Given a vector of per-locus allele frequencies,
`founders.founder_haplotypes_from_AFs()` builds a
`DenseHaplotypeArray`:


In [ ]:
import xftsim as xft
from xftsim import founders

afs = [0.0, 1.0, 0.5]
founders.founder_haplotypes_from_AFs(n=10, afs=afs)


For convenience, `founders.founder_haplotypes_uniform_AFs()` will
uniformly sample `m` allele frequencies between `minMAF` and
`1 - minMAF`:


In [ ]:
founders.founder_haplotypes_uniform_AFs(n=10, m=10, minMAF=0.05)

:::{note}
When founder constructors create variant metadata without an
explicit chromosome layout, `xftsim` evenly divides variants between
(up to) 22 chromosomes.
:::

Of course you can construct a `DenseHaplotypeArray` directly from a
3-D `(n, m, 2)` numpy array:


In [ ]:
import numpy as np

genos = np.random.binomial(1, 0.3, size=(100, 200, 2)).astype(np.int8)
xft.struct.DenseHaplotypeArray(genos)


## Haplotypes from external datasets

`xftsim` currently supports PLINK binary (bfile) and VCF/sgkit
formats. (We hope to add bgen / plink2 pfile support in the future.)

### PLINK bfiles

:::{warning}
The PLINK bfile format is inherently diploid and will break phasing —
haplotypes at heterozygous loci are assigned randomly. For phased
data, prefer the VCF / sgkit path or a GRG (see below).
:::

We can read a bfile using
`founders.founder_haplotypes_from_plink_bfile()`. Example using the
data packaged with `pandas_plink`:


In [ ]:
from pandas_plink import get_data_folder
from os.path import join

pdat = founders.founder_haplotypes_from_plink_bfile(
    join(get_data_folder(), 'chr*.bed')
)
pdat


The legacy lazy-dask backing has been removed; the call returns a
materialised `DenseHaplotypeArray` directly.

To write a haplotype array back out as PLINK1, use
`xftsim.io.read_plink1_as_pseudohaplotypes` for input and the
corresponding writer for output (see the API reference for the
current list).

### VCF / sgkit

VCF support is provided through `sgkit`. The recommended workflow is:

1. Convert the VCF (or a subset of it) to a [Zarr](https://zarr.readthedocs.io)
   store with `sgkit.io.vcf.vcf_to_zarr()`.
2. Lazy-load it with `sgkit.load_dataset()`.
3. Convert to an `xftsim`-compatible array with
   `xftsim.founders.founder_haplotypes_from_sgkit_dataset()` (which is
   re-exported by `xftsim.io.haplotypes_from_sgkit_dataset`).
4. Save the result to npz for future runs with
   `xft.io.save_haplotypes_npz`.

### GRG (graph-based representation)

`xftsim` also supports msprime-derived
[GRG](https://github.com/jonathanmatthewdaniels/GRG)
graphs for memory-efficient haplotype storage. Use
`xftsim.founders.founder_haplotypes_from_msprime_grg()` to simulate
new founders via msprime + GRG, or `xftsim.io.load_grg()` to load an
existing GRG file. Both return a `GraphHaplotypeOperator` that
implements the same `matvec` / `rmatvec` / `standardized_matvec` API
as `DenseHaplotypeArray`, so it drops straight into a simulation.

## Haplotypes from npz

The preferred on-disk format for saving and loading haplotype arrays
in v0.9 is npz. Save with `xft.io.save_haplotypes_npz()` and load with
`xft.io.load_haplotypes_npz()` — both round-trip the genotypes plus the
full `SampleMeta` / `VariantMeta`.
